# 01 — Master Metadata Sheet

Build a single, clean, sample-level metadata table joining:
1. **ODK mosquito data** — species, sex, blood-fed status, sample_id (ep-barcode)
2. **ODK morpho-ID sessions** — barcode, household-id, date, collector
3. **ODK household data** — GPS coordinates, building type (from `ckmr_sampling` and `ckmr_uvlt`)
4. **SeqArt tracking** — plate position
5. **Genomic data** — which samples are in the DArTseq Report files

Output: `data/master_metadata.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data")
ODK = Path("../sampling/odk")

## 1. Load ODK data

In [2]:
# Individual mosquito records (species, sex, blood-fed, ep-barcode)
mosquito = pd.read_csv(ODK / "ckmr_morpho_id-mosquito_data.csv")
print(f"Mosquito records: {len(mosquito)}")
print(f"Columns: {list(mosquito.columns)}")
mosquito.head(3)

Mosquito records: 994
Columns: ['note', 'morph_id', 'morph_id_other', 'mosquito_sex', 'blood_fed_status', 'notes', 'sample_id', 'PARENT_KEY', 'KEY']


,note,morph_id,morph_id_other,mosquito_sex,blood_fed_status,notes,sample_id,PARENT_KEY,KEY
0,NaN,anopheles_funestus,NaN,male,NaN,NaN,ep0000844050,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49/mosq...
1,NaN,anopheles_funestus,NaN,male,NaN,NaN,ep0000912124,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49/mosq...
2,NaN,anopheles_funestus,NaN,female,unfed,NaN,ep0000843959,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49/mosq...


In [3]:
# Morpho-ID sessions (barcode, household-id, date, collector)
morpho = pd.read_csv(ODK / "ckmr_morpho_id.csv")
print(f"Morpho-ID sessions: {len(morpho)}")
morpho.head(3)

Morpho-ID sessions: 285


,SubmissionDate,initialise-date_collected,initialise-collector,initialise-barcode,initialise-household-id,initialise-collect_bool,end_note,meta-instanceID,KEY,SubmitterID,SubmitterName,AttachmentsPresent,AttachmentsExpected,Status,ReviewState,DeviceID,Edits
0,2025-03-12T07:31:00.815Z,2025-03-12T00:00:00.000+03:00,alphonse_omondi,282,A1-10-6 -Out,no,NaN,uuid:f78a8f1a-deea-4491-ad10-af321a49b66b,uuid:f78a8f1a-deea-4491-ad10-af321a49b66b,519,ckmr-team-member,0,0,NaN,NaN,collect:Fb2Iw2jh93cwJatm,NaN
1,2025-03-12T07:29:38.785Z,2025-03-12T00:00:00.000+03:00,calvince_walloh,296,C2-10-2-Out,no,NaN,uuid:e5fd9f62-979f-4d54-952e-1905b6a6c956,uuid:e5fd9f62-979f-4d54-952e-1905b6a6c956,519,ckmr-team-member,0,0,NaN,NaN,collect:kcC8bfCjw04TiIMa,NaN
2,2025-03-12T07:28:57.919Z,2025-03-12T00:00:00.000+03:00,victoria_nguyai,283,A1-10-6-In,yes,NaN,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,519,ckmr-team-member,0,0,NaN,NaN,collect:kcC8bfCjw04TiIMa,NaN


In [4]:
# Household-level data with GPS — two sources
sampling = pd.read_csv(ODK / "ckmr_sampling.csv")
uvlt = pd.read_csv(ODK / "ckmr_uvlt.csv")
print(f"ckmr_sampling records: {len(sampling)}")
print(f"ckmr_uvlt records: {len(uvlt)}")

ckmr_sampling records: 29
ckmr_uvlt records: 132


## 2. Join mosquito data to morpho-ID sessions

The `PARENT_KEY` in mosquito data links to `KEY` in morpho-ID sessions. This gives us barcode, household-id, date, and collector for each mosquito.

In [5]:
# Select useful columns from morpho sessions
morpho_cols = morpho[["initialise-date_collected", "initialise-collector", 
                       "initialise-barcode", "initialise-household-id", "KEY"]].copy()
morpho_cols.columns = ["date_collected", "collector", "barcode", "household_id", "session_key"]

# Join mosquito → morpho session
df = mosquito.merge(morpho_cols, left_on="PARENT_KEY", right_on="session_key", how="left")
print(f"After join: {len(df)} rows")
print(f"Unmatched (no session): {df['session_key'].isna().sum()}")
df.head(3)

After join: 994 rows
Unmatched (no session): 0


,note,morph_id,morph_id_other,mosquito_sex,blood_fed_status,notes,sample_id,PARENT_KEY,KEY,date_collected,collector,barcode,household_id,session_key
0,NaN,anopheles_funestus,NaN,male,NaN,NaN,ep0000844050,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49/mosq...,2025-03-12T00:00:00.000+03:00,victoria_nguyai,283,A1-10-6-In,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49
1,NaN,anopheles_funestus,NaN,male,NaN,NaN,ep0000912124,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49/mosq...,2025-03-12T00:00:00.000+03:00,victoria_nguyai,283,A1-10-6-In,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49
2,NaN,anopheles_funestus,NaN,female,unfed,NaN,ep0000843959,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49/mosq...,2025-03-12T00:00:00.000+03:00,victoria_nguyai,283,A1-10-6-In,uuid:5fdc7521-9884-4a7b-99f2-887d7e6d3e49


## 3. Build GPS lookup from household data

GPS comes from two forms: `ckmr_sampling` (29 records, day 1) and `ckmr_uvlt` (132 records, most days). We'll build a unified GPS lookup keyed by **barcode** (the cup/tube barcode assigned at collection).

In [6]:
# GPS from ckmr_sampling: one barcode per household
gps_sampling = sampling[["collection_end-barcode", "initialise-gps-Latitude", 
                          "initialise-gps-Longitude", "initialise-gps-Altitude",
                          "initialise-gps-Accuracy",
                          "house_data-roof_type", "house_data-wall_type", 
                          "house_data-eaves_type", "house_data-unique_code",
                          "house_data-day", "house_data-team"]].copy()
gps_sampling.columns = ["barcode", "latitude", "longitude", "altitude", "gps_accuracy",
                         "roof_type", "wall_type", "eaves_type", "unique_code", "day", "team"]
gps_sampling["barcode"] = gps_sampling["barcode"].astype(str).str.strip()
gps_sampling["gps_source"] = "ckmr_sampling"
print(f"GPS from sampling: {len(gps_sampling)} records")

# GPS from ckmr_uvlt: two barcodes per household (indoor + outdoor)
gps_indoor = uvlt[["collection_end-barcode_indoor", "initialise-gps-Latitude",
                    "initialise-gps-Longitude", "initialise-gps-Altitude",
                    "initialise-gps-Accuracy",
                    "house_data-roof_type", "house_data-wall_type",
                    "house_data-eaves_type", "house_data-unique_code_indoor",
                    "house_data-day", "house_data-team"]].copy()
gps_indoor.columns = ["barcode", "latitude", "longitude", "altitude", "gps_accuracy",
                       "roof_type", "wall_type", "eaves_type", "unique_code", "day", "team"]

gps_outdoor = uvlt[["collection_end-barcode_outdoor", "initialise-gps-Latitude",
                     "initialise-gps-Longitude", "initialise-gps-Altitude",
                     "initialise-gps-Accuracy",
                     "house_data-roof_type", "house_data-wall_type",
                     "house_data-eaves_type", "house_data-unique_code_outdoor",
                     "house_data-day", "house_data-team"]].copy()
gps_outdoor.columns = ["barcode", "latitude", "longitude", "altitude", "gps_accuracy",
                        "roof_type", "wall_type", "eaves_type", "unique_code", "day", "team"]

gps_uvlt = pd.concat([gps_indoor, gps_outdoor], ignore_index=True)
gps_uvlt["barcode"] = gps_uvlt["barcode"].astype(str).str.strip()
gps_uvlt["gps_source"] = "ckmr_uvlt"
print(f"GPS from uvlt: {len(gps_uvlt)} records")

# Combine, preferring uvlt (more complete) but filling gaps from sampling
gps_all = pd.concat([gps_uvlt, gps_sampling], ignore_index=True)
gps_all = gps_all.drop_duplicates(subset="barcode", keep="first")
print(f"Combined GPS lookup: {len(gps_all)} unique barcodes")
gps_all.head(3)

GPS from sampling: 29 records
GPS from uvlt: 264 records
Combined GPS lookup: 293 unique barcodes


,barcode,latitude,longitude,altitude,gps_accuracy,roof_type,wall_type,eaves_type,unique_code,day,team,gps_source
0,301,0.105319,34.184452,1231.706421,4.647,metal,mud,open_eaves,j2-10-5-in,10,2,ckmr_uvlt
1,283,0.102494,34.195899,1216.289673,4.429,metal,mud,open_eaves,a1-10-6-in,10,1,ckmr_uvlt
2,284,0.101818,34.195506,1201.268188,4.949,thatch,mud,open_eaves,a1-10-5-in,10,1,ckmr_uvlt


## 4. Join GPS to mosquito data via barcode

In [7]:
# Ensure barcode types match
df["barcode"] = df["barcode"].astype(str).str.strip()

df = df.merge(gps_all, on="barcode", how="left")
print(f"After GPS join: {len(df)} rows")
print(f"With GPS: {df['latitude'].notna().sum()}")
print(f"Without GPS: {df['latitude'].isna().sum()}")
print(f"\nGPS coverage: {df['latitude'].notna().mean():.1%}")

After GPS join: 994 rows
With GPS: 989
Without GPS: 5

GPS coverage: 99.5%


## 5. Add genomic data presence flag

Extract sample IDs from the DArTseq SilicoDArT file header (row 7) and flag which mosquitoes have genomic data.

In [8]:
# Read only row 7 (header=6) to get sample IDs — skip the 6 metadata rows
header_row = pd.read_csv(DATA / "Report_DMo25-3087_SilicoDArT_1.csv", 
                          header=6, nrows=0)
genomic_sample_ids = set(header_row.columns[15:])  # first 15 cols are marker metadata
print(f"Samples in genomic data: {len(genomic_sample_ids)}")
print(f"Example IDs: {list(genomic_sample_ids)[:5]}")

# Flag in metadata
df["in_genomic_data"] = df["sample_id"].isin(genomic_sample_ids)
print(f"\nIn both ODK and genomic: {df['in_genomic_data'].sum()}")
print(f"In ODK only: {(~df['in_genomic_data']).sum()}")

Samples in genomic data: 894
Example IDs: ['ep0000911965', 'ep0000916649', 'ep0000922038', 'ep0000918759', 'ep0000915468']

In both ODK and genomic: 901
In ODK only: 93


In [9]:
# Check for genomic samples NOT in ODK
odk_sample_ids = set(df["sample_id"].dropna())
genomic_only = genomic_sample_ids - odk_sample_ids
print(f"Samples in genomic data but NOT in ODK: {len(genomic_only)}")
if genomic_only:
    print(f"  IDs: {genomic_only}")

Samples in genomic data but NOT in ODK: 1
  IDs: {'ep0000911796'}


## 6. Add SeqArt plate positions

In [10]:
# The tracking file has plate positions but odk_barcode is empty.
# Instead, reconstruct plate positions from the DArTseq header rows.
header_lines = pd.read_csv(DATA / "Report_DMo25-3087_SilicoDArT_1.csv", 
                            header=None, nrows=7)

# Rows: 0=order, 1=batch, 2=plate, 3=well_row, 4=well_col, 5=species, 6=col_names
# Sample columns start at index 15
plate_info = pd.DataFrame({
    "sample_id": header_lines.iloc[6, 15:].values,
    "plate": header_lines.iloc[2, 15:].values,
    "well_row": header_lines.iloc[3, 15:].values,
    "well_col": header_lines.iloc[4, 15:].values,
})
plate_info["well"] = plate_info["well_row"].astype(str) + plate_info["well_col"].astype(str)
print(f"Plate positions: {len(plate_info)}")
print(f"Plates: {sorted(plate_info['plate'].unique())}")

df = df.merge(plate_info[["sample_id", "plate", "well_row", "well_col", "well"]], 
              on="sample_id", how="left")
print(f"Samples with plate info: {df['plate'].notna().sum()}")

Plate positions: 894
Plates: ['1', '10', '2', '3', '4', '5', '6', '7', '8', '9']
Samples with plate info: 901


## 7. Parse household-ID components

The household-id encodes `Team-Day-Household[-Position]`. Parse these out for grouping and analysis.

In [11]:
# Parse household_id: "A1-10-6-In" → team=A1, day=10, house=6, position=In
def parse_household_id(hid):
    if pd.isna(hid):
        return pd.Series({"hid_team": np.nan, "hid_day": np.nan, 
                          "hid_house": np.nan, "hid_position": np.nan})
    parts = str(hid).strip().split("-")
    if len(parts) >= 3:
        team = parts[0]
        day = parts[1]
        house = parts[2]
        position = parts[3] if len(parts) > 3 else np.nan
        return pd.Series({"hid_team": team, "hid_day": day, 
                          "hid_house": house, "hid_position": position})
    return pd.Series({"hid_team": np.nan, "hid_day": np.nan, 
                      "hid_house": np.nan, "hid_position": np.nan})

hid_parsed = df["household_id"].apply(parse_household_id)
df = pd.concat([df, hid_parsed], axis=1)
print("Parsed household-id components:")
print(f"  Teams: {sorted(df['hid_team'].dropna().unique())}")
print(f"  Days: {sorted(df['hid_day'].dropna().unique())}")
print(f"  Failed to parse: {df['hid_team'].isna().sum()}")

Parsed household-id components:
  Teams: ['A1', 'A2', 'A3', 'AI', 'C1', 'C2', 'C3', 'I3', 'J', 'J2', 'L3', 'S1', 'S2', 'S3', 'V1']
  Days: ['1', '10', '2', '3', '34', '4', '5', '6', '7', '8', '9']
  Failed to parse: 5


## 8. QC checks

In [12]:
print("=" * 60)
print("QC REPORT")
print("=" * 60)

# 1. Duplicate sample IDs
dup_ids = df["sample_id"].value_counts()
dups = dup_ids[dup_ids > 1]
print(f"\n1. Duplicate sample_ids: {len(dups)}")
if len(dups) > 0:
    print(f"   {dups.to_dict()}")

# 2. Missing values summary
print(f"\n2. Missing values:")
for col in ["sample_id", "morph_id", "mosquito_sex", "date_collected", 
            "barcode", "household_id", "latitude", "longitude"]:
    n_miss = df[col].isna().sum()
    pct = n_miss / len(df) * 100
    print(f"   {col}: {n_miss} ({pct:.1f}%)")

# 3. Species distribution (expect mostly An. funestus)
print(f"\n3. Species distribution:")
print(df["morph_id"].value_counts().to_string())

# 4. Species vs genomic data
print(f"\n4. Species of samples IN genomic data:")
print(df[df["in_genomic_data"]]["morph_id"].value_counts().to_string())

print(f"\n5. Species of samples NOT in genomic data:")
print(df[~df["in_genomic_data"]]["morph_id"].value_counts().to_string())

QC REPORT

1. Duplicate sample_ids: 6
   {'ep0000919729': 3, 'ep0000919001': 3, 'ep0000909970': 2, 'ep0000915423': 2, 'ep0000915142': 2, 'ep0000916071': 2}

2. Missing values:
   sample_id: 0 (0.0%)
   morph_id: 0 (0.0%)
   mosquito_sex: 0 (0.0%)
   date_collected: 0 (0.0%)
   barcode: 0 (0.0%)
   household_id: 0 (0.0%)
   latitude: 5 (0.5%)
   longitude: 5 (0.5%)

3. Species distribution:
morph_id
anopheles_funestus      914
anopheles_coustani       47
anopheles_gambiae_sl     23
other                    10

4. Species of samples IN genomic data:
morph_id
anopheles_funestus      897
anopheles_gambiae_sl      3
anopheles_coustani        1

5. Species of samples NOT in genomic data:
morph_id
anopheles_coustani      46
anopheles_gambiae_sl    20
anopheles_funestus      17
other                   10


In [13]:
# 6. GPS coordinate sanity check (should be near Lake Kanyaboli: ~0.07°N, ~34.18°E)
gps_df = df.dropna(subset=["latitude", "longitude"])
print(f"6. GPS sanity check ({len(gps_df)} records with GPS):")
print(f"   Latitude  range: {gps_df['latitude'].min():.4f} to {gps_df['latitude'].max():.4f} (expect ~0.07)")
print(f"   Longitude range: {gps_df['longitude'].min():.4f} to {gps_df['longitude'].max():.4f} (expect ~34.18)")

outliers = gps_df[
    (gps_df["latitude"] < -1) | (gps_df["latitude"] > 1) |
    (gps_df["longitude"] < 33) | (gps_df["longitude"] > 35)
]
print(f"   GPS outliers (outside Kenya): {len(outliers)}")

# 7. Date range
print(f"\n7. Collection dates:")
print(f"   {df['date_collected'].dropna().unique()}")

# 8. Sex distribution  
print(f"\n8. Sex distribution:")
print(df["mosquito_sex"].value_counts().to_string())

# 9. Blood-fed status
print(f"\n9. Blood-fed status:")
print(df["blood_fed_status"].value_counts().to_string())

6. GPS sanity check (989 records with GPS):
   Latitude  range: 0.0697 to 0.1054 (expect ~0.07)
   Longitude range: 34.1663 to 34.2132 (expect ~34.18)
   GPS outliers (outside Kenya): 0

7. Collection dates:
   <ArrowStringArray>
['2025-03-12T00:00:00.000+03:00', '2025-03-11T00:00:00.000+03:00',
 '2025-03-07T00:00:00.000+03:00', '2025-03-06T00:00:00.000+03:00',
 '2025-03-05T00:00:00.000+03:00', '2025-03-04T00:00:00.000+03:00',
 '2025-02-28T00:00:00.000+03:00', '2025-02-27T00:00:00.000+03:00',
 '2025-02-26T00:00:00.000+03:00', '2025-02-25T00:00:00.000+03:00',
 '2025-02-24T00:00:00.000+03:00']
Length: 11, dtype: str

8. Sex distribution:
mosquito_sex
female    679
male      315

9. Blood-fed status:
blood_fed_status
unfed     513
gravid    120
fed        68


### Investigate duplicate sample_ids

In [14]:
# Show all rows for duplicate sample_ids
dup_mask = df["sample_id"].isin(dups.index)
dup_df = df[dup_mask].sort_values("sample_id")[
    ["sample_id", "morph_id", "mosquito_sex", "blood_fed_status", 
     "barcode", "household_id", "date_collected", "collector"]
]
dup_df

,sample_id,morph_id,mosquito_sex,blood_fed_status,barcode,household_id,date_collected,collector
323,ep0000909970,anopheles_funestus,male,NaN,186,S3-7-2-In,2025-03-06T00:00:00.000+03:00,stephen_owino
992,ep0000909970,anopheles_funestus,female,unfed,513,T3,2025-02-24T00:00:00.000+03:00,laban_adero
694,ep0000915142,anopheles_funestus,female,gravid,90,S3-4-3-Out,2025-02-28T00:00:00.000+03:00,calvince_walloh
717,ep0000915142,anopheles_funestus,male,NaN,81,V1-3-4-Out,2025-02-27T00:00:00.000+03:00,calvince_walloh
639,ep0000915423,anopheles_funestus,male,NaN,135,A1-5-4-In,2025-03-04T00:00:00.000+03:00,calvince_walloh
640,ep0000915423,anopheles_funestus,male,NaN,135,A1-5-4-In,2025-03-04T00:00:00.000+03:00,calvince_walloh
968,ep0000916071,anopheles_funestus,male,unfed,17,J2-1-3,2025-02-25T00:00:00.000+03:00,victoria_nguyai
969,ep0000916071,anopheles_funestus,male,unfed,17,J2-1-3,2025-02-25T00:00:00.000+03:00,victoria_nguyai
633,ep0000919001,anopheles_funestus,female,unfed,131,S3-5-1-In,2025-03-04T00:00:00.000+03:00,calvince_walloh
989,ep0000919001,anopheles_gambiae_sl,female,unfed,511,J2-,2025-02-24T00:00:00.000+03:00,alphonse_omondi


### Investigate mosquitoes without GPS

In [15]:
# Which mosquitoes are missing GPS?
no_gps = df[df["latitude"].isna()][
    ["sample_id", "morph_id", "barcode", "household_id", "date_collected", "in_genomic_data"]
]
no_gps

,sample_id,morph_id,barcode,household_id,date_collected,in_genomic_data
989,ep0000919001,anopheles_gambiae_sl,511,J2-,2025-02-24T00:00:00.000+03:00,True
990,ep0000919729,anopheles_gambiae_sl,509,J2-3,2025-02-24T00:00:00.000+03:00,True
991,ep0000919729,anopheles_gambiae_sl,509,J2-3,2025-02-24T00:00:00.000+03:00,True
992,ep0000909970,anopheles_funestus,513,T3,2025-02-24T00:00:00.000+03:00,True
993,ep0000919001,anopheles_funestus,513,T3,2025-02-24T00:00:00.000+03:00,True


## 9. Clean up and export

In [16]:
# Select and rename final columns
final_cols = [
    "sample_id", "morph_id", "morph_id_other", "mosquito_sex", "blood_fed_status",
    "date_collected", "collector", "barcode", "household_id",
    "hid_team", "hid_day", "hid_house", "hid_position",
    "latitude", "longitude", "altitude", "gps_accuracy", "gps_source",
    "unique_code", "roof_type", "wall_type", "eaves_type",
    "plate", "well_row", "well_col", "well",
    "in_genomic_data", "notes"
]

# Only keep columns that exist
final_cols = [c for c in final_cols if c in df.columns]
master = df[final_cols].copy()

# Sort by plate position for consistency
master = master.sort_values(["plate", "well_row", "well_col"], na_position="last").reset_index(drop=True)

print(f"Master metadata: {len(master)} rows × {len(master.columns)} columns")
print(f"\nColumns: {list(master.columns)}")
master.head()

Master metadata: 994 rows × 28 columns

Columns: ['sample_id', 'morph_id', 'morph_id_other', 'mosquito_sex', 'blood_fed_status', 'date_collected', 'collector', 'barcode', 'household_id', 'hid_team', 'hid_day', 'hid_house', 'hid_position', 'latitude', 'longitude', 'altitude', 'gps_accuracy', 'gps_source', 'unique_code', 'roof_type', 'wall_type', 'eaves_type', 'plate', 'well_row', 'well_col', 'well', 'in_genomic_data', 'notes']


,sample_id,morph_id,morph_id_other,mosquito_sex,blood_fed_status,date_collected,collector,barcode,household_id,hid_team,...,unique_code,roof_type,wall_type,eaves_type,plate,well_row,well_col,well,in_genomic_data,notes
0,ep0000917698,anopheles_funestus,NaN,male,unfed,2025-02-25T00:00:00.000+03:00,julius_ochieng,501,S3-1-7,S3,...,s3-1-7,metal,mud,open_eaves,1,A,1,A1,True,NaN
1,ep0000844321,anopheles_funestus,NaN,female,unfed,2025-02-25T00:00:00.000+03:00,julius_ochieng,506,S3-1-6,S3,...,s3-1-6,metal,cement,open_eaves,1,A,10,A10,True,NaN
2,ep0000910700,anopheles_funestus,NaN,female,unfed,2025-02-25T00:00:00.000+03:00,victoria_nguyai,19,C2-1-4,C2,...,c2-1-4,metal,mud,open_eaves,1,A,11,A11,True,NaN
3,ep0000915878,anopheles_funestus,NaN,male,unfed,2025-02-25T00:00:00.000+03:00,laban_adero,505,I3-1-3,I3,...,l3-1-3,metal,mud,open_eaves,1,A,12,A12,True,NaN
4,ep0000846383,anopheles_funestus,NaN,male,unfed,2025-02-25T00:00:00.000+03:00,julius_ochieng,501,S3-1-7,S3,...,s3-1-7,metal,mud,open_eaves,1,A,2,A2,True,NaN


In [17]:
# Export
master.to_csv(DATA / "master_metadata.csv", index=False)
print(f"Saved to data/master_metadata.csv")
print(f"\nSummary:")
print(f"  Total mosquitoes: {len(master)}")
print(f"  With genomic data: {master['in_genomic_data'].sum()}")
print(f"  With GPS: {master['latitude'].notna().sum()}")
print(f"  With both genomic + GPS: {(master['in_genomic_data'] & master['latitude'].notna()).sum()}")

Saved to data/master_metadata.csv

Summary:
  Total mosquitoes: 994
  With genomic data: 901
  With GPS: 989
  With both genomic + GPS: 896
